# Contrastive Explanation Results

Visualizes the Instantiation I ablation results from `scripts/out/journal/summary_contrastive.csv`.

Run `scripts/run_experiments.py` first to generate the data.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

REPO = Path('/Users/ssuresh/gambit')
CSV  = REPO / 'scripts' / 'out' / 'journal' / 'summary_contrastive.csv'

df = pd.read_csv(CSV)

# Derived columns
METHODS = ['base_evidence', 'naive_contrastive', 'optimized']
METHOD_LABELS = {'base_evidence': 'Base', 'naive_contrastive': 'Naive', 'optimized': 'CDEA'}
EVIDENCE_LABELS = {'gradcam': 'Grad-CAM', 'ig': 'IG'}
datasets = df['dataset'].unique().tolist()
evidence_types = df['evidence'].unique().tolist()

print(f'Datasets: {datasets}')
print(f'Evidence: {evidence_types}')
print(f'Seeds: n={df["n_seeds"].iloc[0]}')
df.head()

## 1. Main Results Table

Optimized (CDEA) metrics vs base evidence, one row per (dataset, evidence).

In [ ]:
base = df[df['method'] == 'base_evidence'].set_index(['dataset', 'evidence'])
opt  = df[df['method'] == 'optimized'].set_index(['dataset', 'evidence'])

rows = []
for ds in datasets:
    for ev in evidence_types:
        if (ds, ev) not in base.index or (ds, ev) not in opt.index:
            continue
        b = base.loc[(ds, ev)]
        o = opt.loc[(ds, ev)]
        ov_red = (b['overlap_mean'] - o['overlap_mean']) / max(abs(b['overlap_mean']), 1e-8) * 100
        sp_red = (b['sparse_mean'] - o['sparse_mean']) / max(abs(b['sparse_mean']), 1e-8) * 100
        rows.append({
            'Dataset': ds, 'Evidence': EVIDENCE_LABELS.get(ev, ev),
            'Suff ↑': f"{o['suff_mean']:.4f}",
            'Margin ↑': f"{o['margin_mean']:.4f}",
            'Overlap ↓': f"{o['overlap_mean']:.4f}",
            'Overlap ↓% vs Base': f"{ov_red:+.1f}%",
            'Sparse ↓': f"{o['sparse_mean']:.4f}",
            'Sparse ↓% vs Base': f"{sp_red:+.1f}%",
        })

main_table = pd.DataFrame(rows)
main_table.style.set_caption('CDEA (optimized) — main results').hide(axis='index')

## 2. Overlap by Method — Grouped Bar Chart

Shows how CDEA drives overlap toward zero vs base evidence and naive contrastive.  
Overlap is an unnormalized pairwise dot-product sum — lower is better.

In [ ]:
n_ev = len(evidence_types)
n_ds = len(datasets)
fig, axes = plt.subplots(1, n_ev, figsize=(5 * n_ev, 4), sharey=False)
if n_ev == 1:
    axes = [axes]

colors = {'base_evidence': '#6baed6', 'naive_contrastive': '#fd8d3c', 'optimized': '#31a354'}
x = np.arange(n_ds)
width = 0.25

for ax, ev in zip(axes, evidence_types):
    for i, method in enumerate(METHODS):
        vals = [
            df[(df['dataset'] == ds) & (df['evidence'] == ev) & (df['method'] == method)]['overlap_mean'].values
            for ds in datasets
        ]
        heights = [v[0] if len(v) > 0 else 0 for v in vals]
        errs = [
            df[(df['dataset'] == ds) & (df['evidence'] == ev) & (df['method'] == method)]['overlap_std'].values
            for ds in datasets
        ]
        errs = [e[0] if len(e) > 0 else 0 for e in errs]
        bars = ax.bar(x + i * width, heights, width, label=METHOD_LABELS[method],
                      color=colors[method], yerr=errs, capsize=3, error_kw={'linewidth': 1})

    ax.set_title(EVIDENCE_LABELS.get(ev, ev), fontsize=12)
    ax.set_xticks(x + width)
    ax.set_xticklabels(datasets)
    ax.set_ylabel('Overlap (↓ better)')
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.set_ylim(bottom=0)

fig.suptitle('Pairwise Mask Overlap by Method', fontsize=13, y=1.02)
plt.tight_layout()
out = REPO / 'examples' / 'out' / 'plot_overlap_by_method.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print('Saved to', out)

## 3. Overlap Reduction % — Heatmap

Percent overlap reduction of CDEA over base evidence.  
Higher is better — the headline result.

In [ ]:
heat = np.zeros((len(datasets), len(evidence_types)))
for i, ds in enumerate(datasets):
    for j, ev in enumerate(evidence_types):
        b_rows = df[(df['dataset'] == ds) & (df['evidence'] == ev) & (df['method'] == 'base_evidence')]
        o_rows = df[(df['dataset'] == ds) & (df['evidence'] == ev) & (df['method'] == 'optimized')]
        if b_rows.empty or o_rows.empty:
            heat[i, j] = float('nan')
            continue
        b_ov = b_rows['overlap_mean'].values[0]
        o_ov = o_rows['overlap_mean'].values[0]
        heat[i, j] = (b_ov - o_ov) / max(abs(b_ov), 1e-8) * 100

fig, ax = plt.subplots(figsize=(max(3, len(evidence_types) * 1.5), max(2.5, len(datasets) * 0.7)))
im = ax.imshow(heat, cmap='YlGn', aspect='auto', vmin=0, vmax=100)
plt.colorbar(im, ax=ax, label='Overlap reduction (%) ↑')

ax.set_xticks(range(len(evidence_types)))
ax.set_xticklabels([EVIDENCE_LABELS.get(e, e) for e in evidence_types])
ax.set_yticks(range(len(datasets)))
ax.set_yticklabels(datasets)

for i in range(len(datasets)):
    for j in range(len(evidence_types)):
        v = heat[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.1f}%', ha='center', va='center',
                    fontsize=11, fontweight='bold',
                    color='black' if v < 60 else 'white')

ax.set_title('Overlap Reduction: CDEA vs Base Evidence', fontsize=12)
plt.tight_layout()
out = REPO / 'examples' / 'out' / 'plot_overlap_reduction_heatmap.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print('Saved to', out)

## 4. Sufficiency–Overlap Tradeoff

Each point is a (dataset, evidence, method) combination.  
Ideal: high sufficiency (right) + low overlap (bottom) — bottom-right corner.  
CDEA should cluster bottom-right vs base/naive clustering top-right.

In [ ]:
markers = {'mnist': 'o', 'cifar10': 's', 'pets': '^', 'stanford_dogs': 'D'}
ev_colors = {'gradcam': '#2166ac', 'ig': '#d6604d'}
method_alpha = {'base_evidence': 0.4, 'naive_contrastive': 0.65, 'optimized': 1.0}
method_size  = {'base_evidence': 60,  'naive_contrastive': 80,  'optimized': 130}

fig, ax = plt.subplots(figsize=(7, 5))

for _, row in df.iterrows():
    ds = row['dataset']
    ev = row['evidence']
    method = row['method']
    suff = row['suff_mean']
    ov   = row['overlap_mean']
    ax.scatter(suff, ov,
               marker=markers.get(ds, 'o'),
               color=ev_colors.get(ev, 'gray'),
               alpha=method_alpha.get(method, 0.7),
               s=method_size.get(method, 80),
               edgecolors='white' if method == 'optimized' else 'none',
               linewidths=1.5,
               zorder=3 if method == 'optimized' else 2)

# Legend: methods
for method, label in METHOD_LABELS.items():
    ax.scatter([], [], marker='o', color='gray',
               alpha=method_alpha[method], s=method_size[method],
               label=label,
               edgecolors='white' if method == 'optimized' else 'none', linewidths=1.5)
# Legend: evidence
for ev, color in ev_colors.items():
    if ev in evidence_types:
        ax.scatter([], [], marker='o', color=color, s=80, label=EVIDENCE_LABELS.get(ev, ev), alpha=0.8)
# Legend: datasets
for ds, mk in markers.items():
    if ds in datasets:
        ax.scatter([], [], marker=mk, color='gray', s=80, label=ds, alpha=0.7)

ax.set_xlabel('Sufficiency ↑ (target logit under keep intervention)')
ax.set_ylabel('Overlap ↓ (pairwise mask dot-product sum)')
ax.set_title('Sufficiency–Overlap Tradeoff\n(CDEA = large filled, Base = small faded)')
ax.legend(fontsize=8, loc='upper right', ncol=2)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
out = REPO / 'examples' / 'out' / 'plot_suff_overlap_tradeoff.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print('Saved to', out)

## 5. Margin Improvement — CDEA vs Base

Contrastive margin = `z_k − max(z_foil)` under the keep intervention.  
Bars show the delta: `margin(CDEA) − margin(base)`. Positive = CDEA improves contrastive separation.

In [ ]:
fig, ax = plt.subplots(figsize=(max(5, len(datasets) * 1.4), 4))

x = np.arange(len(datasets))
width = 0.35
ev_colors_list = ['#2166ac', '#d6604d', '#4daf4a', '#984ea3']

for j, ev in enumerate(evidence_types):
    deltas = []
    for ds in datasets:
        b = df[(df['dataset'] == ds) & (df['evidence'] == ev) & (df['method'] == 'base_evidence')]
        o = df[(df['dataset'] == ds) & (df['evidence'] == ev) & (df['method'] == 'optimized')]
        if b.empty or o.empty:
            deltas.append(0)
        else:
            deltas.append(o['margin_mean'].values[0] - b['margin_mean'].values[0])
    offset = (j - (len(evidence_types) - 1) / 2) * width
    bars = ax.bar(x + offset, deltas, width,
                  label=EVIDENCE_LABELS.get(ev, ev),
                  color=ev_colors_list[j % len(ev_colors_list)])
    for bar, v in zip(bars, deltas):
        ax.text(bar.get_x() + bar.get_width() / 2,
                v + (0.0002 if v >= 0 else -0.0005),
                f'{v:+.4f}', ha='center', va='bottom' if v >= 0 else 'top',
                fontsize=8)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(datasets)
ax.set_ylabel('Margin delta (CDEA − Base) ↑')
ax.set_title('Contrastive Margin Improvement: CDEA vs Base Evidence')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
out = REPO / 'examples' / 'out' / 'plot_margin_improvement.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print('Saved to', out)

## 6. Per-Batch Metric Distribution

Box plots of per-batch overlap across methods, for each (dataset, evidence) combination.  
Shows variance across batches — useful for understanding optimizer stability.

In [ ]:
import glob

# Load all per-batch CSVs that match the current datasets/evidence
per_batch_frames = []
pattern = str(REPO / 'scripts' / 'out' / 'ablation_*_seed*_per_batch.csv')
for fpath in sorted(glob.glob(pattern)):
    fname = Path(fpath).stem  # e.g. ablation_cifar10_gradcam_seed0_per_batch
    parts = fname.split('_')
    # parts: ['ablation', dataset, evidence, 'seed0', 'per', 'batch']
    if len(parts) < 4:
        continue
    dataset_name = parts[1]
    evidence_name = parts[2]
    seed_tag = parts[3]  # e.g. seed0
    tmp = pd.read_csv(fpath)
    tmp['dataset'] = dataset_name
    tmp['evidence'] = evidence_name
    tmp['seed'] = seed_tag
    per_batch_frames.append(tmp)

if not per_batch_frames:
    print('No per-batch CSVs found. Run run_experiments.py first.')
else:
    pb = pd.concat(per_batch_frames, ignore_index=True)
    combos = pb[['dataset', 'evidence']].drop_duplicates().values.tolist()
    n_combos = len(combos)
    fig, axes = plt.subplots(1, n_combos, figsize=(4 * n_combos, 4), sharey=False)
    if n_combos == 1:
        axes = [axes]

    for ax, (ds, ev) in zip(axes, combos):
        sub = pb[(pb['dataset'] == ds) & (pb['evidence'] == ev)]
        data = [sub[sub['method'] == m]['overlap'].dropna().values for m in METHODS]
        labels = [METHOD_LABELS[m] for m in METHODS]
        bp = ax.boxplot(data, labels=labels, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 2})
        for patch, method in zip(bp['boxes'], METHODS):
            patch.set_facecolor({'base_evidence': '#6baed6',
                                  'naive_contrastive': '#fd8d3c',
                                  'optimized': '#31a354'}[method])
            patch.set_alpha(0.7)
        ax.set_title(f'{ds} / {EVIDENCE_LABELS.get(ev, ev)}')
        ax.set_ylabel('Overlap ↓')
        ax.grid(axis='y', linestyle='--', alpha=0.4)

    fig.suptitle('Per-Batch Overlap Distribution by Method', fontsize=13, y=1.02)
    plt.tight_layout()
    out = REPO / 'examples' / 'out' / 'plot_overlap_distribution.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved to', out)